PePy

Author: Julia K. Varga <jvarga92@gmail.com>  
License: BSD 3 clause  
Code Repository: https://github.com/gezmi/pepy

In [1]:
import pandas as pd
pd.set_option('display.width', 600)
pd.set_option('display.max_columns', 8)

# Working with Confidence Data

PePy can load confidence metrics produced by structure prediction methods — AlphaFold2/ColabFold, AlphaFold3, and ChAI.

Confidence data includes:
- **PAE matrix** — Predicted Aligned Error between all residue pairs
- **iPTM** — interface predicted TM-score
- **pTM** — predicted TM-score
- **pLDDT** — per-residue confidence (stored in B-factor column of structure files)

This tutorial shows how to load and use confidence data from each format.

In [2]:
from pepy import ProteinComplex

## AlphaFold2 / ColabFold

AF2 stores confidence data in a JSON file alongside the PDB structure. PePy auto-discovers it by converting the structure filename:

- `*_unrelaxed_*` → `*_scores_*`
- `.pdb` → `.json`

In [3]:
af2 = ProteinComplex.from_file(
    '../../pepy/tests/data/1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb'
)
af2.identify_chains()

# Load confidence — auto-discovers the JSON
loaded = af2.load_confidence_data()
print(f'Confidence loaded: {loaded}')
af2.get_confidence_summary()

Binder chain(s): B, receptor chain(s): A
Confidence loaded: True


{'status': 'confidence_loaded',
 'has_pae_matrix': True,
 'has_iptm': True,
 'has_ptm': True,
 'iptm': 0.85,
 'ptm': 0.78,
 'combined_confidence': 0.84}

## AlphaFold3

AF3 uses two JSON files:
- `*_full_data_*.json` — contains the PAE matrix
- `*_summary_confidences_*.json` — contains iPTM, pTM, etc.

Both are merged automatically:

In [4]:
af3 = ProteinComplex.from_file('../../pepy/tests/data/fold_1ycr_af3_model_0.cif')
af3.identify_chains()

loaded = af3.load_confidence_data()
print(f'Confidence loaded: {loaded}')
af3.get_confidence_summary()

Binder chain(s): B, receptor chain(s): A
Confidence loaded: True


{'status': 'confidence_loaded',
 'has_pae_matrix': True,
 'has_iptm': True,
 'has_ptm': True,
 'iptm': 0.76,
 'ptm': 0.74,
 'combined_confidence': 0.76}

## ChAI

ChAI stores confidence data in a NumPy `.npz` file alongside the PDB structure.
PePy matches them by the model index in the filename:

In [5]:
chai = ProteinComplex.from_file('../../pepy/tests/data/pred.model_idx_0.pdb')
chai.identify_chains()

loaded = chai.load_confidence_data()
print(f'Confidence loaded: {loaded}')
chai.get_confidence_summary()

Binder chain(s): B, receptor chain(s): A
Confidence loaded: True


{'status': 'confidence_loaded',
 'has_pae_matrix': False,
 'has_iptm': True,
 'has_ptm': False,
 'iptm': 0.826}

## Explicit confidence file path

If auto-discovery doesn't find your file (e.g. non-standard naming), you can provide the path explicitly:

In [6]:
af2.load_confidence_data(
    confidence_file_path='../../pepy/tests/data/1ycr_af2_55d19_scores_rank_001_alphafold2_multimer_v3_model_1_seed_000.json'
)
af2.get_confidence_summary()

{'status': 'confidence_loaded',
 'has_pae_matrix': True,
 'has_iptm': True,
 'has_ptm': True,
 'iptm': 0.85,
 'ptm': 0.78,
 'combined_confidence': 0.84}

## Accessing the Raw Data

After loading, the raw confidence data is available on the complex object:

In [7]:
print(f"iPTM: {af2.confidence_data['iptm']}")
print(f"pTM:  {af2.confidence_data['ptm']}")
print(f"Combined (0.8*iPTM + 0.2*pTM): {af2.confidence_data['confidence']}")

iPTM: 0.85
pTM:  0.78
Combined (0.8*iPTM + 0.2*pTM): 0.84


### PAE matrix

The PAE matrix is a pandas DataFrame — rows and columns correspond to CA atom indices in the structure:

In [8]:
pae = af2.confidence_data['pae_matrix']
print(f'PAE matrix shape: {pae.shape}')
pae.iloc[:5, :5]

PAE matrix shape: (124, 124)


,0,1,2,3,4
0,0.75,2.19,4.22,6.35,8.41
1,1.29,0.75,2.27,4.43,6.01
2,3.57,1.22,0.75,1.61,3.82
3,5.74,3.30,1.11,0.75,1.85
4,7.84,5.90,4.38,1.41,0.75


### pLDDT values

Per-residue pLDDT is stored in the B-factor column of the structure, not in the confidence file:

In [9]:
ca_atoms = af2.df['ATOM'][af2.df['ATOM']['atom_name'] == 'CA']
ca_atoms[['chain_id', 'residue_number', 'residue_name', 'b_factor']].head(10)

,chain_id,residue_number,residue_name,b_factor
1,A,1,SER,34.94
7,A,2,GLN,41.31
16,A,3,ILE,44.78
24,A,4,PRO,48.78
31,A,5,ALA,44.88
36,A,6,SER,45.53
42,A,7,GLU,51.19
51,A,8,GLN,57.00
60,A,9,GLU,73.38
69,A,10,THR,80.56


## Interface Confidence Metrics

`calculate_confidence_metrics()` combines interface residues with the confidence data to compute interface-specific metrics.

This requires both the interface and confidence data to be loaded first:

In [10]:
# Calculate interface first
af2.calculate_interface()

# Then compute confidence metrics for the interface
metrics = af2.calculate_confidence_metrics()
metrics

{'status': 'calculated',
 'avg_plddt_interface': 91.68,
 'max_plddt_interface': 98.19,
 'interface_pae': 1.97,
 'min_interface_pae': 0.95,
 'iptm': 0.85,
 'ptm': 0.78,
 'combined_confidence': 0.84,
 'has_interface_metrics': True}

Key metrics:

| Metric | Description |
|--------|-------------|
| `avg_plddt_interface` | Mean pLDDT of binder interface CA atoms |
| `max_plddt_interface` | Max pLDDT of binder interface CA atoms |
| `interface_pae` | Median PAE between binder–receptor interface residues |
| `min_interface_pae` | Min PAE between binder–receptor interface residues |
| `iptm` | Interface predicted TM-score (global) |
| `ptm` | Predicted TM-score (global) |
| `combined_confidence` | 0.8 × iPTM + 0.2 × pTM |

### Interface PAE submatrix

You can also extract the PAE submatrix for just the interface residues. This uses the `ca_index` column that maps each residue to its position in the PAE matrix:

In [11]:
atom_df = af2.df['ATOM']
pae = af2.confidence_data['pae_matrix']

# Get CA indices for interface residues
binder_ca = atom_df[
    (atom_df['chain_id'].isin(af2.binder_chains)) &
    (atom_df['residue_number'].isin(af2.interface_residues_binder)) &
    (atom_df['atom_name'] == 'CA')
]
receptor_ca = atom_df[
    (atom_df['chain_id'].isin(af2.receptor_chains)) &
    (atom_df['residue_number'].isin(af2.interface_residues_receptor)) &
    (atom_df['atom_name'] == 'CA')
]

# Extract submatrix
binder_idx = binder_ca['ca_index'].tolist()
receptor_idx = receptor_ca['ca_index'].tolist()
pae_sub = pae.iloc[binder_idx, receptor_idx]

print(f'Interface PAE submatrix: {pae_sub.shape}')
print(f'Median: {pae_sub.median().median():.2f}')
print(f'Min:    {pae_sub.min().min():.2f}')
pae_sub

Interface PAE submatrix: (10, 13)
Median: 1.97
Min:    0.95


,34,37,41,44,...,76,79,82,83
110,13.20,13.22,10.47,12.60,...,8.92,14.24,14.55,14.55
111,7.97,7.29,4.04,3.60,...,4.57,7.16,7.57,8.91
112,2.29,1.79,1.46,1.39,...,1.46,2.10,2.01,2.23
113,1.60,1.29,1.08,1.17,...,1.11,1.47,1.47,1.58
116,1.51,1.22,1.10,1.22,...,1.02,1.30,1.26,1.40
117,1.34,1.13,0.99,1.15,...,0.97,1.28,1.22,1.37
119,2.09,1.66,1.52,1.83,...,1.49,1.62,1.93,1.94
120,2.02,1.69,1.71,2.12,...,1.55,1.59,1.74,1.77
121,3.74,3.04,3.66,4.44,...,2.53,2.66,3.49,3.43
122,5.39,5.65,9.93,9.98,...,6.15,4.62,6.60,6.18


## Adding Support for New Formats

The confidence file discovery uses pattern lists defined at the top of `pepy/io/confidence.py`. To add support for a new prediction method:

1. Add filename patterns to `AF2_STRIP_PATTERNS` / `AF2_REPLACEMENTS` (for PDB-based outputs)
2. Or add new glob patterns in `_find_af3_or_chai()` (for CIF/NPZ-based outputs)
3. If the file uses new metric key names, add them to `METRIC_KEYS`

In [12]:
from pepy.io.confidence import AF2_STRIP_PATTERNS, AF2_REPLACEMENTS, METRIC_KEYS

print('Strip patterns:', AF2_STRIP_PATTERNS)
print('Replacements:', AF2_REPLACEMENTS)
print('Metric keys:', METRIC_KEYS)

Strip patterns: ['_superimpos', '_superimposed', '_super', '_truncated', '_align_inter_pymol']
Replacements: [('.pdb', '.json'), ('_unrelaxed_', '_scores_'), ('_relaxed_', '_scores_')]
Metric keys: {'pae': ['predicted_aligned_error', 'pae', 'PAE'], 'iptm': ['iptm', 'i_ptm', 'ipTM', 'iPTM'], 'ptm': ['ptm', 'PTM', 'pTM', 'pTm'], 'plddt': ['plddt', 'confidence', 'bfactor']}
